In [25]:
import sys, os
ROOT = "/Users/fserracrespi/Desktop/PD_PROJECT_UOFL" 
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)
from utils.Decode import decoder_DF
from utils.dataframe_cols import cols_asignacion2
from utils.Prog_df import check_progression
from utils.Prog_df import progression_csv_upgrade
from utils.Prog_df import check_progression_multi
from utils.Prog_df import progression_multi_csv_upgrade
from utils.Prog_df import visit_csv_upgrade
from utils.Prog_df import check_visits
import json
from collections import defaultdict
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import kurtosis, skew
import re
import math

In [26]:
path2='/Users/fserracrespi/Desktop/PD_Project_UofL/DATA_COLLECTION/12_09_2025/Study_Docs/Data___Databases/DATA/Data_Dictionary_-_Harmonized_12Sep2025.csv'
path3='/Users/fserracrespi/Desktop/PD_Project_UofL/DATA_COLLECTION/12_09_2025/Study_Docs/Data___Databases/DATA/Code_List_-_Harmonized_12Sep2025.csv'
orden_visitas = ["BL", "V04", "V06", "V08", "V10", "V12"]
code_cols=pd.read_csv(path2, dtype=str)
code_rows=pd.read_csv(path3, dtype=str)
PATNOs = pd.read_csv('/Users/fserracrespi/Desktop/PD_Project_UofL/PD_DATA/PD_CSV_CLEAN/Relevant_presnece_PATNO', dtype=str)['PATNO'].tolist()
print(f'Total PATNOs to process: {len(PATNOs)}')

Total PATNOs to process: 1056


# MoCA

In [27]:
moca_df=pd.read_csv('/Users/fserracrespi/Desktop/PD_Project_UofL/DATA_COLLECTION/12_09_2025/Non-Motor Assestments/Neuropsychological/DATA/Montreal_Cognitive_Assessment__MoCA__12Sep2025.csv', dtype=str)
moca_df=moca_df[moca_df['PATNO'].isin(PATNOs)]
moca_df=decoder_DF(moca_df, code_rows, code_cols, module='MOCA')
moca_df=moca_df.loc[moca_df['Visit ID'].isin(["BL", "V04", "V06", "V08", "V10", "V12"])]
moca_df.dropna(how='any',inplace=True)
moca_df.head(3)

/Users/fserracrespi/Desktop/PD_PROJECT_UOFL/utils/Decode.py:81: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df[col].replace('NaN', np.nan, inplace=True)


,Record ID,PATNO,Visit ID,Page Name,Site Aware Date,Alternating Trail Making,Visuoconstructional Skills (Cube),Visuoconstructional Skills (Clock Cont),Visuoconstructional Skills (Clock Num),Visuoconstructional Skills (Clock Hands),...,Delayed Recall - Red,Orientation - Date,Orientation - Month,Orientation - Year,Orientation - Day,Orientation - Place,Orientation - City,MoCA Total Score,Date of original data entry,Date of most recent update to record
10,341440701,3001,V04,Montreal Cognitive Assessment (MoCA),03/2012,1,1,1,1,1,...,1,1,1,1,1,1,1,30,04/2012,2020-06-25 16:04:33.0
11,398208101,3001,V06,Montreal Cognitive Assessment (MoCA),05/2013,1,1,1,1,1,...,1,1,1,1,1,1,1,30,05/2013,2020-06-25 16:04:34.0
12,448720301,3001,V08,Montreal Cognitive Assessment (MoCA),04/2014,0,1,1,1,1,...,1,1,1,1,1,1,1,29,07/2014,2020-06-25 16:04:34.0


In [28]:
moca_df['Attention - Vigilance'].replace({'1 = 0 or 1 errors': 1, '0 = 2 or more errors': 0}, inplace=True)
moca_df['Attention - Serial 7s'].replace({'3 = 4 or 5 correct':3, '1 = 1 correct':1, '2 = 2 or 3 correct':2,
       '0 = 0 correct':0}, inplace=True)
moca_df['Verbal Fluency'].replace({'1 = 11 or more words':1, '0 = 10 or less words':0}, inplace=True)


/var/folders/vr/_v9wq92941bflv75jsh8kq800000gn/T/ipykernel_27389/2026881498.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  moca_df['Attention - Vigilance'].replace({'1 = 0 or 1 errors': 1, '0 = 2 or more errors': 0}, inplace=True)
/var/folders/vr/_v9wq92941bflv75jsh8kq800000gn/T/ipykernel_27389/2026881498.py:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavi

In [29]:
moca_dict = {
    "VISUOSPATIAL_EXECUTIVE": [
        "Alternating Trail Making",
        "Visuoconstructional Skills (Cube)",
        "Visuoconstructional Skills (Clock Cont)",
        "Visuoconstructional Skills (Clock Num)",
        "Visuoconstructional Skills (Clock Hands)"
    ],
    "NAMING": [
        "Naming - Lion",
        "Naming - Rhino",
        "Naming - Camel"
    ],
    "ATTENTION": [
        "Attention - Forward Digit Span",
        "Attention - Backward Digit Span",
        "Attention - Vigilance",
        "Attention - Serial 7s"
    ],
    "LANGUAGE": [
        "Sentence Repetition",
        "Verbal Fluency"

    ],
    "ABSTRACTION": [
        "Abstraction"
    ],
    "DELAYED_RECALL": [
        "Delayed Recall - Face",
        "Delayed Recall - Velvet",
        "Delayed Recall - Church",
        "Delayed Recall - Daisy",
        "Delayed Recall - Red"
    ],
    "ORIENTATION": [
        "Orientation - Date",
        "Orientation - Month",
        "Orientation - Year",
        "Orientation - Day",
        "Orientation - Place",
        "Orientation - City"
    ]
}

list_cols = []
for key in moca_dict.keys():
    cols = moca_dict[key]
    for col in cols:
        moca_df[col] = pd.to_numeric(moca_df[col], errors='coerce')

    moca_df[key] = moca_df[cols].sum(axis=1)

relevant_cols = ['PATNO', 'Visit ID', 'MoCA Total Score'] + list(moca_dict.keys())
moca_df = moca_df[relevant_cols]
moca_df

,PATNO,Visit ID,MoCA Total Score,VISUOSPATIAL_EXECUTIVE,NAMING,ATTENTION,LANGUAGE,ABSTRACTION,DELAYED_RECALL,ORIENTATION
10,3001,V04,30,5,3,6,3,2,5,6
11,3001,V06,30,5,3,6,3,2,5,6
12,3001,V08,29,4,3,6,3,2,5,6
13,3001,V10,29,5,3,6,2,2,5,6
14,3001,V12,29,4,3,6,3,2,5,6
...,...,...,...,...,...,...,...,...,...,...
16945,446733,BL,29,5,3,6,3,2,4,6
16950,452611,BL,26,5,3,6,1,2,2,6
16954,466157,BL,26,5,3,6,0,1,5,6
16955,476236,BL,25,3,3,5,1,2,4,6


# SCOPA-AUT DATA

In [30]:
scopa_data=pd.read_csv('/Users/fserracrespi/Desktop/PD_Project_UofL/DATA_COLLECTION/12_09_2025/Non-Motor Assestments/Autonomic Test/SCOPA-AUT_12Sep2025.csv',dtype=str)
print( scopa_data.shape)
print( scopa_data['PATNO'].nunique())
scopa_data = decoder_DF(scopa_data, code_rows, code_cols, module='SCOPAAUT')
scopa_data=scopa_data[scopa_data['PATNO'].isin(PATNOs)]
scopa_data=scopa_data.loc[scopa_data['Visit ID'].isin(["BL", "V04", "V06", "V08", "V10", "V12"])]
scopa_data.head()

(18132, 43)
4339


/Users/fserracrespi/Desktop/PD_PROJECT_UOFL/utils/Decode.py:81: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df[col].replace('NaN', np.nan, inplace=True)


,Record ID,PATNO,Visit ID,Page Name,Site Aware Date,Source of Information,SCOPA Item 1,SCOPA Item 2,SCOPA Item 3,SCOPA Item 4,...,SCOPA Item 26a,SCOPA Item 26a med,SCOPA Item 26b,SCOPA Item 26b med,SCOPA Item 26c,SCOPA Item 26c med,SCOPA Item 26d,SCOPA Item 26d med,Date of original data entry,Date of most recent update to record
9,278749101,3001,BL,SCOPA-AUT,03/2011,Participant,never,never,never,never,...,No,NaN,No,NaN,No,NaN,Yes,Rosacea,03/2011,2020-06-25 16:04:31.0
11,341444201,3001,V04,SCOPA-AUT,03/2012,Participant,never,never,never,never,...,No,NaN,Yes,vesicare,Yes,lisinopril,Yes,rosacea,04/2012,2020-06-25 16:04:33.0
12,396138701,3001,V06,SCOPA-AUT,05/2013,Participant,sometimes,never,never,never,...,No,NaN,Yes,vesicare,Yes,lisinopril,Yes,"azelaic acid, metronidazole",05/2013,2020-06-25 16:04:34.0
13,448723201,3001,V08,SCOPA-AUT,04/2014,Participant,sometimes,sometimes,sometimes,sometimes,...,No,NaN,Yes,vesicare,Yes,lisinopril,Yes,"finacea, metronidazole",07/2014,2020-06-25 16:04:34.0
14,516494401,3001,V10,SCOPA-AUT,04/2015,Participant,sometimes,never,never,never,...,No,NaN,Yes,vesicare,Yes,lisinopril,Yes,"finacea,metronidazole,clelopirox dlaminde",04/2015,2020-06-25 16:04:35.0


In [31]:
scopa_aut_dict = {
    "Gastrointestinal": [
        "SCOPA Item 1",
        "SCOPA Item 2",
        "SCOPA Item 3",
        "SCOPA Item 4",
        "SCOPA Item 5",
        "SCOPA Item 6",
        "SCOPA Item 7"
    ],
    "Urinario": [
        "SCOPA Item 8",
        "SCOPA Item 9",
        "SCOPA Item 10",
        "SCOPA Item 11",
        "SCOPA Item 12",
        "SCOPA Item 13"
    ],
    "Cardiovascular": [
        "SCOPA Item 14",
        "SCOPA Item 15",
        "SCOPA Item 16"
    ],
    "Termorregulatorio": [
        "SCOPA Item 17",
        "SCOPA Item 18",
        "SCOPA Item 20",
        "SCOPA Item 21"
    ],
    "Pupillomotor": [
        "SCOPA Item 19"
    ],
    "Sexual_Hombres": [
        "SCOPA Item 22",
        "SCOPA Item 23"
    ],
    "Sexual_Mujeres": [
        "SCOPA Item 24",
        "SCOPA Item 25"
    ]
}

list_cols=[]
for key in scopa_aut_dict.keys():
    list_cols=list_cols+scopa_aut_dict[key]
scopa_data=scopa_data[['PATNO','Visit ID']+list_cols]

scopa_data['Use_catheter'] = np.where(
    (scopa_data[scopa_aut_dict['Urinario']] == 'use catheter').any(axis=1),
    True,
    False
)
scopa_data[list_cols] = scopa_data[list_cols].replace({'use catheter': np.nan})

scopa_data.replace(to_replace={
    "never": 0,
    "sometimes": 1,
    "regularly": 2,
    "often": 3,
    "not applicable": np.nan
}, inplace=True)

scopa_data
scopa_data.dropna(how='any', subset=list_cols[:-4], inplace=True)
scopa_data.isna().sum()

s22 = scopa_data['SCOPA Item 22']
s23 = scopa_data['SCOPA Item 23']
s24 = scopa_data['SCOPA Item 24']
s25 = scopa_data['SCOPA Item 25']

# Válido si:
cond_hombre_valido = s22.notna() & s23.notna() & s24.isna() & s25.isna()
cond_mujer_valida  = s24.notna() & s25.notna() & s22.isna() & s23.isna()

mask_valido = cond_hombre_valido | cond_mujer_valida
scopa_data = scopa_data[mask_valido]

scopa_data["Sexo_inferido"] = np.where(cond_hombre_valido[mask_valido], "Hombre", "Mujer")
scopa_data.isna().sum()

scopa_data['Score_Gastrointestinal'] = scopa_data[scopa_aut_dict['Gastrointestinal']].sum(axis=1)
scopa_data['Score_Urinario'] = scopa_data[scopa_aut_dict['Urinario']].sum(axis=1)
scopa_data['Score_Cardiovascular'] = scopa_data[scopa_aut_dict['Cardiovascular']].sum(axis=1)
scopa_data['Score_Termorregulatorio'] = scopa_data[scopa_aut_dict['Termorregulatorio']].sum(axis=1)
scopa_data['Score_Pupillomotor'] = scopa_data[scopa_aut_dict['Pupillomotor']].sum(axis=1)

scopa_data['Score_Sexual'] = np.where(
    scopa_data['Sexo_inferido'] == 'Hombre',
    scopa_data[scopa_aut_dict['Sexual_Hombres']].sum(axis=1),
    scopa_data[scopa_aut_dict['Sexual_Mujeres']].sum(axis=1)
)

scopa_data['SCOPA-AUT Total Score'] = scopa_data[['Score_Gastrointestinal', 'Score_Urinario', 'Score_Cardiovascular', 'Score_Termorregulatorio', 'Score_Pupillomotor', 'Score_Sexual']].sum(axis=1)
scopa_data.drop(columns=list_cols+['Sexo_inferido','Use_catheter'], inplace=True)




/var/folders/vr/_v9wq92941bflv75jsh8kq800000gn/T/ipykernel_27389/3585895876.py:55: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  scopa_data.replace(to_replace={


# Epworth_Sleepiness_Scale

In [32]:
ep_df=pd.read_csv('/Users/fserracrespi/Desktop/PD_Project_UofL/DATA_COLLECTION/12_09_2025/Non-Motor Assestments/Sleep_Disorder/DATA/Epworth_Sleepiness_Scale_12Sep2025.csv', dtype=str)
ep_df=ep_df[ep_df['PATNO'].isin(PATNOs)]
ep_df=decoder_DF(ep_df, code_rows, code_cols, module='EPWORTH')
ep_df=ep_df.loc[ep_df['Visit ID'].isin(["BL", "V04", "V06", "V08", "V10", "V12"])]
ep_df.replace(to_replace={
    "Would never doze": 0,
    "Slight chance of dozing": 1,
    "Moderate chance of dozing": 2,
    "High chance of dozing": 3
}, inplace=True)
ep_df=ep_df.loc[ep_df['Source of Information']=='Participant',:]
ep_df.fillna(1, inplace=True)

ep_df['Epworth Sleepiness Scale Score'] = ep_df[
    ['Sitting and reading', 
     'Watching TV',
     'Sitting, inactive in a public place',
     'As a passenger in a car for an hour',
     'Lying down to rest in the afternoon',
     'Sitting and talking to someone',
     'Sitting quietly after lunch',
     'In a car, while stopped in traffic']
].sum(axis=1)

ep_df.drop(columns=['Record ID','Page Name','Site Aware Date','Source of Information','Date of original data entry','Date of most recent update to record'], inplace=True)
ep_df

/Users/fserracrespi/Desktop/PD_PROJECT_UOFL/utils/Decode.py:81: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df[col].replace('NaN', np.nan, inplace=True)
/var/folders/vr/_v9wq92941bflv75jsh8kq800000gn/T/ipykernel_27389/1843847314.py:5: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  ep_df.replace(to_

,PATNO,Visit ID,Sitting and reading,Watching TV,"Sitting, inactive in a public place",As a passenger in a car for an hour,Lying down to rest in the afternoon,Sitting and talking to someone,Sitting quietly after lunch,"In a car, while stopped in traffic",Epworth Sleepiness Scale Score
9,3001,BL,1,3,0.0,0,2,0.0,0,0,6.0
11,3001,V04,0,1,0.0,0,2,0.0,0,0,3.0
12,3001,V06,1,2,1.0,1,2,0.0,0,0,7.0
13,3001,V08,1,2,0.0,1,1,0.0,0,0,5.0
14,3001,V10,1,2,1.0,0,2,0.0,0,0,6.0
...,...,...,...,...,...,...,...,...,...,...,...
18123,431490,BL,0,1,0.0,0,1,0.0,0,0,2.0
18137,446160,BL,1,0,0.0,0,3,0.0,0,0,4.0
18138,446733,BL,1,2,1.0,1,1,0.0,1,0,7.0
18143,452611,BL,2,2,1.0,2,2,1.0,2,1,13.0


# REM

In [33]:
rem_df=pd.read_csv('/Users/fserracrespi/Desktop/PD_Project_UofL/DATA_COLLECTION/12_09_2025/Non-Motor Assestments/Sleep_Disorder/DATA/REM_Sleep_Behavior_Disorder_Questionnaire_12Sep2025.csv', dtype=str)
rem_df=rem_df[rem_df['PATNO'].isin(PATNOs)]
rem_df=decoder_DF(rem_df, code_rows, code_cols, module='REMSLEEP')
rem_df=rem_df.loc[rem_df['Visit ID'].isin(["BL", "V04", "V06", "V08", "V10", "V12"])]
rem_df=rem_df.dropna(how='any')


/Users/fserracrespi/Desktop/PD_PROJECT_UOFL/utils/Decode.py:81: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df[col].replace('NaN', np.nan, inplace=True)


In [34]:
rem_df.drop(columns=['Record ID','Site Aware Date','Page Name','Date of original data entry','Date of most recent update to record','other'], inplace=True)
rem_df.replace(to_replace={'Yes':1, 'No':0}, inplace=True)
rem_df=rem_df.loc[rem_df['Source of Information']=='Participant',:]
rem_df.drop(columns=['Source of Information'], inplace=True)
rem_df

/var/folders/vr/_v9wq92941bflv75jsh8kq800000gn/T/ipykernel_27389/3896469800.py:2: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  rem_df.replace(to_replace={'Yes':1, 'No':0}, inplace=True)


,PATNO,Visit ID,Vivid Dreams,Aggressive or Action-packed dreams,nocturnal behaviour,move arms/legs during sleep,hurt bed partner,speaking in sleep,sudden limb movements,complex movements,...,remember dreams,sleep is disturbed,Stroke,Head trauma,Parkinsonism,Restless Leg Syndrome (RLS),Narcolepsy,Depression,Epilepsy,Inflammatory disease of the brain
9,3001,BL,1,0,0,1,0,0,0,0,...,1,0,0,0,1,0,0,0,0,0
11,3001,V04,1,0,0,0,0,1,0,0,...,1,0,0,0,1,0,0,1,0,0
12,3001,V06,1,1,0,0,0,1,1,0,...,1,0,0,0,1,0,0,0,0,0
13,3001,V08,1,0,0,1,0,1,1,1,...,0,0,0,0,1,0,0,0,0,0
14,3001,V10,1,0,0,1,0,1,1,0,...,1,0,0,0,1,0,0,1,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
18135,431490,BL,0,0,0,0,0,0,0,0,...,0,0,0,0,1,0,0,0,0,0
18149,446160,BL,1,0,0,0,0,1,0,0,...,0,0,0,0,1,0,0,0,0,0
18150,446733,BL,1,0,0,0,0,1,0,1,...,1,0,0,0,0,0,0,0,0,0
18155,452611,BL,0,0,0,0,0,0,0,0,...,0,1,0,0,1,0,0,0,0,0


# Modified Semantic Fluency

In [35]:
semantic_df=pd.read_csv('/Users/fserracrespi/Desktop/PD_Project_UofL/DATA_COLLECTION/12_09_2025/Non-Motor Assestments/Neuropsychological/DATA/Modified_Semantic_Fluency_12Sep2025.csv',dtype=str)
semantic_df=semantic_df[semantic_df['PATNO'].isin(PATNOs)]
semantic_df=decoder_DF(semantic_df, code_rows, code_cols, module='SFT')
semantic_df=semantic_df.loc[semantic_df['Visit ID'].isin(["BL", "V04", "V06", "V08", "V10", "V12"])]
relevant_cols=['PATNO','Visit ID','Scaled score']
semantic_df=semantic_df[relevant_cols]
semantic_df.rename(columns={'Scaled score':'Semantic Fluency Scaled Score'}, inplace=True)

semantic_df

/Users/fserracrespi/Desktop/PD_PROJECT_UOFL/utils/Decode.py:81: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df[col].replace('NaN', np.nan, inplace=True)


,PATNO,Visit ID,Semantic Fluency Scaled Score
9,3001,BL,10
10,3001,V04,14
11,3001,V06,9
12,3001,V08,14
13,3001,V10,13
...,...,...,...
16272,446733,BL,11
16277,452611,BL,9
16281,466157,BL,10
16282,476236,BL,3


# Letter Sequencing

In [36]:
letter_df=pd.read_csv('/Users/fserracrespi/Desktop/PD_Project_UofL/DATA_COLLECTION/12_09_2025/Non-Motor Assestments/Neuropsychological/DATA/Letter_-_Number_Sequencing_12Sep2025.csv',dtype=str)
letter_df=letter_df[letter_df['PATNO'].isin(PATNOs)]
letter_df=decoder_DF(letter_df, code_rows, code_cols, module='LNSPD')
letter_df=letter_df.loc[letter_df['Visit ID'].isin(["BL", "V04", "V06", "V08", "V10", "V12"])]
relevant_cols=['PATNO','Visit ID','Scaled score']
letter_df=letter_df[relevant_cols]
letter_df.rename(columns={'Scaled score':'Letter-Number Sequencing Scaled Score'}, inplace=True)
letter_df


/Users/fserracrespi/Desktop/PD_PROJECT_UOFL/utils/Decode.py:81: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df[col].replace('NaN', np.nan, inplace=True)


,PATNO,Visit ID,Letter-Number Sequencing Scaled Score
9,3001,BL,17
10,3001,V04,17
11,3001,V06,18
12,3001,V08,15
13,3001,V10,18
...,...,...,...
16227,446733,BL,14
16232,452611,BL,11
16236,466157,BL,7
16237,476236,BL,6


# Benton_Judgement_of_Line_Orientation

In [37]:
benton_df=pd.read_csv('/Users/fserracrespi/Desktop/PD_Project_UofL/DATA_COLLECTION/12_09_2025/Non-Motor Assestments/Neuropsychological/DATA/Benton_Judgement_of_Line_Orientation_12Sep2025.csv',dtype=str)
benton_df=benton_df[benton_df['PATNO'].isin(PATNOs)]
benton_df=decoder_DF(benton_df, code_rows, code_cols, module='BENTONOD')
benton_df=benton_df.loc[benton_df['Visit ID'].isin(["BL", "V04", "V06", "V08", "V10", "V12"])]
relevant_cols=['PATNO','Visit ID','Scaled score adjusted by age and education']
benton_df=benton_df[relevant_cols]
benton_df['Scaled score adjusted by age and education'].fillna(benton_df['Scaled score adjusted by age and education'].astype(float).median(), inplace=True)
benton_df.rename(columns={'Scaled score adjusted by age and education':'Benton Judgement of Line Orientation Scaled Score'}, inplace=True)
benton_df


/Users/fserracrespi/Desktop/PD_PROJECT_UOFL/utils/Decode.py:81: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df[col].replace('NaN', np.nan, inplace=True)
/var/folders/vr/_v9wq92941bflv75jsh8kq800000gn/T/ipykernel_27389/4196565098.py:7: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.


,PATNO,Visit ID,Benton Judgement of Line Orientation Scaled Score
9,3001,BL,15.46
10,3001,V04,11.06
11,3001,V06,15.46
12,3001,V08,12.16
13,3001,V10,13.26
...,...,...,...
16221,446733,BL,15.46
16226,452611,BL,8.68
16230,466157,BL,7.76
16231,476236,BL,5.61


# Symbol_Digit_Modalities_Test

In [38]:
symbol_df=pd.read_csv('/Users/fserracrespi/Desktop/PD_Project_UofL/DATA_COLLECTION/12_09_2025/Non-Motor Assestments/Neuropsychological/DATA/Symbol_Digit_Modalities_Test_12Sep2025.csv',dtype=str)
symbol_df=symbol_df[symbol_df['PATNO'].isin(PATNOs)]
symbol_df=decoder_DF(symbol_df, code_rows, code_cols, module='SDM')
symbol_df=symbol_df.loc[symbol_df['Visit ID'].isin(["BL", "V04", "V06", "V08", "V10", "V12"])]
relevant_cols=['PATNO','Visit ID','Symbol Digit Modalities Total Correct']
symbol_df=symbol_df[relevant_cols]

symbol_df['Symbol Digit Modalities Total Correct'].fillna(symbol_df['Symbol Digit Modalities Total Correct'].astype(float).mean(), inplace=True)

symbol_df.rename(columns={'Symbol Digit Modalities Total Correct':'Symbol Digit Modalities Test Total Correct'}, inplace=True)
symbol_df


/Users/fserracrespi/Desktop/PD_PROJECT_UOFL/utils/Decode.py:81: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df[col].replace('NaN', np.nan, inplace=True)
/var/folders/vr/_v9wq92941bflv75jsh8kq800000gn/T/ipykernel_27389/2113791162.py:8: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.


,PATNO,Visit ID,Symbol Digit Modalities Test Total Correct
9,3001,BL,42
10,3001,V04,36
11,3001,V06,42
12,3001,V08,48
13,3001,V10,48
...,...,...,...
16229,446733,BL,48
16234,452611,BL,42
16238,466157,BL,37
16239,476236,BL,18


# Hopkins_Verbal

In [39]:
hopkins_df=pd.read_csv('/Users/fserracrespi/Desktop/PD_Project_UofL/DATA_COLLECTION/12_09_2025/Non-Motor Assestments/Neuropsychological/DATA/Hopkins_Verbal_Learning_Test_-_Revised_12Sep2025.csv',dtype=str)
hopkins_df=hopkins_df[hopkins_df['PATNO'].isin(PATNOs)]
hopkins_df=decoder_DF(hopkins_df, code_rows, code_cols, module='HVLT')
hopkins_df=hopkins_df.loc[hopkins_df['Visit ID'].isin(["BL", "V04", "V06", "V08", "V10", "V12"])]

hopkins_df['Visit ID'] = pd.Categorical(hopkins_df['Visit ID'], categories=orden_visitas, ordered=True)
hopkins_df=hopkins_df.sort_values(by=['PATNO', 'Visit ID']).reset_index(drop=True)     

columnas_to_convert = [
    'T score for total recall', 
    'T score for delayed recall',
    'T score for retention', 
    'T score for recognition discrimination index'
]

hopkins_df[columnas_to_convert]= hopkins_df[columnas_to_convert].apply(pd.to_numeric, errors='coerce')

relevant_cols=['PATNO','Visit ID'] + columnas_to_convert

hopkins_df=hopkins_df[relevant_cols]
hopkins_df.rename(columns={
    'T score for total recall':'HVLT Total Recall Score',
    'T score for delayed recall':'HVLT Delayed Recall Score',
    'T score for retention':'HVLT Retention Score',
    'T score for recognition discrimination index':'HVLT Recognition Discrimination Index Score'
}, inplace=True)

problematic_patno=hopkins_df.loc[hopkins_df.isna().any(axis=1),'PATNO'].to_list()



# --- Función para rellenar valores faltantes ---
def fill_missing_values(lista_val):
    lista_val = lista_val.copy()
    for i in range(1, len(lista_val)):
        if np.isnan(lista_val[i]):
            lista_val[i] = lista_val[i-1]
    for i in range(len(lista_val)-2, -1, -1):
        if np.isnan(lista_val[i]):
            lista_val[i] = lista_val[i+1]
    return lista_val


# --- Aplicar la función a cada PATNO ---

for patno in problematic_patno:
    mask = hopkins_df['PATNO'] == patno
    for col in ['HVLT Total Recall Score', 'HVLT Delayed Recall Score', 'HVLT Retention Score', 'HVLT Recognition Discrimination Index Score']:
        valores_originales = hopkins_df.loc[mask, col].tolist()
        valores_rellenos = fill_missing_values([float(v) if pd.notna(v) else np.nan for v in valores_originales])
        hopkins_df.loc[mask, col] = valores_rellenos

hopkins_df.isna().sum()
hopkins_df['HVLT Recognition Discrimination Index Score'].fillna(hopkins_df['HVLT Recognition Discrimination Index Score'].mean(), inplace=True)
hopkins_df.isna().sum()




/Users/fserracrespi/Desktop/PD_PROJECT_UOFL/utils/Decode.py:81: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df[col].replace('NaN', np.nan, inplace=True)
/var/folders/vr/_v9wq92941bflv75jsh8kq800000gn/T/ipykernel_27389/1240547914.py:54: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

PATNO                                          0
Visit ID                                       0
HVLT Total Recall Score                        0
HVLT Delayed Recall Score                      0
HVLT Retention Score                           0
HVLT Recognition Discrimination Index Score    0
dtype: int64

# State-Trait_Anxiety

In [40]:
anxiety_df=pd.read_csv('/Users/fserracrespi/Desktop/PD_Project_UofL/DATA_COLLECTION/12_09_2025/Non-Motor Assestments/Neurobehavioral/DATA/State-Trait_Anxiety_Inventory_12Sep2025.csv',dtype=str)
anxiety_df=anxiety_df[anxiety_df['PATNO'].isin(PATNOs)]
anxiety_df=decoder_DF(anxiety_df, code_rows, code_cols, module='STAI')
anxiety_df=anxiety_df.loc[anxiety_df['Visit ID'].isin(["BL", "V04", "V06", "V08", "V10", "V12"])]

anxiety_df['Visit ID'] = pd.Categorical(anxiety_df['Visit ID'], categories=orden_visitas, ordered=True)
anxiety_df=anxiety_df.sort_values(by=['PATNO', 'Visit ID']).reset_index(drop=True)     




/Users/fserracrespi/Desktop/PD_PROJECT_UOFL/utils/Decode.py:81: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df[col].replace('NaN', np.nan, inplace=True)


In [41]:
y1=[ 'STAI Question 1', 'STAI Question 2', 'STAI Question 3', 'STAI Question 4', 'STAI Question 5', 'STAI Question 6',
 'STAI Question 7', 'STAI Question 8', 'STAI Question 9', 'STAI Question 10', 'STAI Question 11', 'STAI Question 12',
 'STAI Question 13', 'STAI Question 14', 'STAI Question 15', 'STAI Question 16', 'STAI Question 17', 'STAI Question 18',
 'STAI Question 19', 'STAI Question 20']
anxiety_df[y1] = anxiety_df[y1].replace({'Not at all': 1, 'Somewhat': 2, 'Moderately so': 3, 'Very much so': 4})
y2=['STAI Question 21', 'STAI Question 22', 'STAI Question 23', 'STAI Question 24', 'STAI Question 25', 'STAI Question 26',
    'STAI Question 27', 'STAI Question 28', 'STAI Question 29', 'STAI Question 30', 'STAI Question 31', 'STAI Question 32',
    'STAI Question 33', 'STAI Question 34', 'STAI Question 35', 'STAI Question 36', 'STAI Question 37', 'STAI Question 38',
    'STAI Question 39', 'STAI Question 40']
anxiety_df[y2] = anxiety_df[y2].replace({'Sometimes':2,'Almost never':1, 'Often':3, 'Almost always':4})


problematic_patno=anxiety_df.loc[anxiety_df.isna().any(axis=1),'PATNO'].to_list()
cols_con_na = anxiety_df.columns[anxiety_df.isna().any()].to_list()


for patno in problematic_patno:
    mask = anxiety_df['PATNO'] == patno
    for col in cols_con_na:
        valores_originales = anxiety_df.loc[mask, col].tolist()
        valores_rellenos = fill_missing_values([float(v) if pd.notna(v) else np.nan for v in valores_originales])
        anxiety_df.loc[mask, col] = valores_rellenos





anxiety_df['STAI PART I Score'] = anxiety_df[y1].sum(axis=1)
anxiety_df['STAI PART II Score'] = anxiety_df[y2].sum(axis=1)
anxiety_df['STAI Total Score'] = anxiety_df[y1+y2].sum(axis=1)

anxiety_df=anxiety_df[['PATNO','Visit ID','STAI PART I Score','STAI PART II Score','STAI Total Score']]
anxiety_df.isna().sum()



/var/folders/vr/_v9wq92941bflv75jsh8kq800000gn/T/ipykernel_27389/2777360230.py:5: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  anxiety_df[y1] = anxiety_df[y1].replace({'Not at all': 1, 'Somewhat': 2, 'Moderately so': 3, 'Very much so': 4})
/var/folders/vr/_v9wq92941bflv75jsh8kq800000gn/T/ipykernel_27389/2777360230.py:10: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  anxiety_df[y2] = anxiety_df[y2].replace({'Sometimes':2,'Almost never':1, 'Often':3, 'Almost always':4})


PATNO                 0
Visit ID              0
STAI PART I Score     0
STAI PART II Score    0
STAI Total Score      0
dtype: int64

# Cognitive_Categorization

In [42]:
cog_df=pd.read_csv('/Users/fserracrespi/Desktop/PD_Project_UofL/DATA_COLLECTION/12_09_2025/Non-Motor Assestments/Other_Cognitive_Assessments/DATA/Cognitive_Categorization_12Sep2025.csv',dtype=str)
cog_df=cog_df[cog_df['PATNO'].isin(PATNOs)]
cog_df=decoder_DF(cog_df, code_rows, code_cols, module='COGCATG')
cog_df=cog_df.loc[cog_df['Visit ID'].isin(["BL", "V04", "V06", "V08", "V10", "V12"])]
cog_df=cog_df.loc[cog_df['Source of Information']=='Participant',:]
cog_df['Visit ID'] = pd.Categorical(cog_df['Visit ID'], categories=orden_visitas, ordered=True)
cog_df=cog_df.sort_values(by=['PATNO', 'Visit ID']).reset_index(drop=True)

relevant_cols=['PATNO','Visit ID','Experienced cognitive decline','Functional impairment due to cognitive','Cognitive State','Level of confidence cognitive diagnosis']
cog_df=cog_df[relevant_cols]

problematic_patno=cog_df.loc[cog_df.isna().any(axis=1),'PATNO'].to_list()
cols_con_na = cog_df.columns[cog_df.isna().any()].to_list()

def fill_miss_text(lista_val):
    lista_val = lista_val.copy()
    # Rellenar hacia adelante
    for i in range(1, len(lista_val)):
        if pd.isna(lista_val[i]):
            lista_val[i] = lista_val[i-1]
    # Rellenar hacia atrás
    for i in range(len(lista_val)-2, -1, -1):
        if pd.isna(lista_val[i]):
            lista_val[i] = lista_val[i+1]
    return lista_val

for patno in problematic_patno:
    mask = cog_df['PATNO'] == patno
    for col in cols_con_na:
        valores_originales = cog_df.loc[mask, col].tolist()
        valores_rellenos = fill_miss_text([
            v if pd.notna(v) else np.nan for v in valores_originales
        ])
        # Asignar los valores corregidos de vuelta al DataFrame
        cog_df.loc[mask, col] = valores_rellenos

cog_df['Level of confidence cognitive diagnosis'].replace({'90% - 100%':'HIGH', '50% - 89%':'MODERATE', '10% - 49%':'MID-LOW','0% - 9%':'LOW'}, inplace=True)

cog_df.isna().sum()
cog_df

/Users/fserracrespi/Desktop/PD_PROJECT_UOFL/utils/Decode.py:81: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df[col].replace('NaN', np.nan, inplace=True)
/var/folders/vr/_v9wq92941bflv75jsh8kq800000gn/T/ipykernel_27389/1325903293.py:37: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

,PATNO,Visit ID,Experienced cognitive decline,Functional impairment due to cognitive,Cognitive State,Level of confidence cognitive diagnosis
0,100001,BL,No,No,Normal Cognition,HIGH
1,100001,V04,No,No,Normal Cognition,HIGH
2,100001,V06,No,No,Normal Cognition,MODERATE
3,100001,V08,No,No,Normal Cognition,MID-LOW
4,100001,V12,Yes,No,Normal Cognition,MODERATE
...,...,...,...,...,...,...
3327,75562,V04,No,No,Normal Cognition,HIGH
3328,75562,V06,No,No,Normal Cognition,HIGH
3329,75562,V08,Yes,No,Mild Cognitive Impairment (MCI),MODERATE
3330,75562,V10,No,No,Normal Cognition,HIGH


# Geriatric Depression Scale 

In [43]:
geriatic_df=pd.read_csv('/Users/fserracrespi/Desktop/PD_Project_UofL/DATA_COLLECTION/12_09_2025/Non-Motor Assestments/Neurobehavioral/DATA/Geriatric_Depression_Scale__Short_Version__12Sep2025.csv',dtype=str)
geriatic_df=geriatic_df[geriatic_df['PATNO'].isin(PATNOs)]
geriatic_df=decoder_DF(geriatic_df, code_rows, code_cols, module='GDSSHORT')
geriatic_df=geriatic_df.loc[geriatic_df['Visit ID'].isin(["BL", "V04", "V06", "V08", "V10", "V12"])]
geriatic_df.fillna(0, inplace=True)
geriatic_df.drop(columns=['Record ID','Page Name','Site Aware Date','Date of original data entry','Date of most recent update to record'], inplace=True)

/Users/fserracrespi/Desktop/PD_PROJECT_UOFL/utils/Decode.py:81: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df[col].replace('NaN', np.nan, inplace=True)


In [44]:
no_1=['Basically satisfied with your life?','In good spirits most of the time?','Feel happy most of the time','Wonderful to be alive now?','Feel full of energy?']
geriatic_df[no_1] = geriatic_df[no_1].replace({'Yes': 0, 'No': 1})
for col in geriatic_df.columns[2:]:
    if col not in no_1:
        geriatic_df[col] = geriatic_df[col].replace({'Yes': 1, 'No': 0})

geriatic_df['GDS Short Score'] = geriatic_df[geriatic_df.columns[4:]].sum(axis=1)
geriatic_df['Has Depression'] = np.where(geriatic_df['GDS Short Score'] >= 5, True, False)
geriatic_df.head()
geriatic_df=geriatic_df[['PATNO','Visit ID','GDS Short Score','Has Depression']]
geriatic_df.head()

/var/folders/vr/_v9wq92941bflv75jsh8kq800000gn/T/ipykernel_27389/3763020445.py:2: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  geriatic_df[no_1] = geriatic_df[no_1].replace({'Yes': 0, 'No': 1})
/var/folders/vr/_v9wq92941bflv75jsh8kq800000gn/T/ipykernel_27389/3763020445.py:5: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  geriatic_df[col] = geriatic_df[col].replace({'Yes': 1, 'No': 0})


,PATNO,Visit ID,GDS Short Score,Has Depression
9,3001,BL,1,False
11,3001,V04,2,False
12,3001,V06,2,False
13,3001,V08,2,False
14,3001,V10,1,False


# QUIP Current Short Not VALID

In [45]:
quip_df=pd.read_csv('/Users/fserracrespi/Desktop/PD_Project_UofL/DATA_COLLECTION/12_09_2025/Non-Motor Assestments/Neurobehavioral/DATA/QUIP-Current-Short_12Sep2025.csv',dtype=str)
quip_df=quip_df[quip_df['PATNO'].isin(PATNOs)]
quip_df=decoder_DF(quip_df, code_rows, code_cols, module='QUIPCS')
quip_df=quip_df.loc[quip_df['Visit ID'].isin(["BL", "V04", "V06", "V08", "V10", "V12"])]
quip_df = quip_df.loc[quip_df['Source of Information']=='Patient', :]
quip_df.drop(columns=['Record ID','Page Name','Site Aware Date','Source of Information','Date of original data entry','Date of most recent update to record'], inplace=True)
quip_df.head()

/Users/fserracrespi/Desktop/PD_PROJECT_UOFL/utils/Decode.py:81: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df[col].replace('NaN', np.nan, inplace=True)


,PATNO,Visit ID,Gambling 1,Gambling 2,Sex 1,Sex 2,Buying 1,Buying 2,Eating 1,Eating 2,Other 1,Other 2,Other 3,Medication 1,Medication 2
9,3001,BL,No,No,No,No,No,No,No,No,Yes,No,No,Not Applicable,Not Applicable
11,3001,V04,No,No,No,No,No,No,No,No,No,No,No,Not Applicable,Not Applicable
12,3001,V06,No,No,No,Yes,No,No,No,No,Yes,No,No,No,No
13,3001,V08,No,No,No,No,No,No,No,No,No,No,No,No,No
14,3001,V10,No,No,No,No,No,No,No,No,No,No,No,No,No


# Lexical Fluency

In [46]:
lexical_df=pd.read_csv('/Users/fserracrespi/Desktop/PD_Project_UofL/DATA_COLLECTION/12_09_2025/Non-Motor Assestments/Neuropsychological/DATA/Lexical_Fluency_12Sep2025.csv',dtype=str)
lexical_df=lexical_df[lexical_df['PATNO'].isin(PATNOs)]
lexical_df=decoder_DF(lexical_df, code_rows, code_cols, module='LEXICAL')
lexical_df=lexical_df.loc[lexical_df['Visit ID'].isin(["BL", "V04", "V06", "V08", "V10", "V12"])]

lexical_df['Visit ID'] = pd.Categorical(lexical_df['Visit ID'], categories=orden_visitas, ordered=True)
lexical_df=lexical_df.sort_values(by=['PATNO', 'Visit ID']).reset_index(drop=True)     

relevant_cols=['PATNO','Visit ID','Scaled score']
lexical_df=lexical_df[relevant_cols]
lexical_df.rename(columns={'Scaled score':'Lexical Fluency Scaled Score'}, inplace=True)
lexical_df['Lexical Fluency Scaled Score']=lexical_df['Lexical Fluency Scaled Score'].astype(float)



lexical_df.isna().sum()
lexical_df.loc[lexical_df['Lexical Fluency Scaled Score'].isna(),:]

/Users/fserracrespi/Desktop/PD_PROJECT_UOFL/utils/Decode.py:81: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df[col].replace('NaN', np.nan, inplace=True)


,PATNO,Visit ID,Lexical Fluency Scaled Score
264,101755,BL,NaN
391,107648,V04,NaN
693,137842,V04,NaN
1484,222261,BL,NaN
1544,235810,V06,NaN
1983,40735,V12,NaN


In [47]:
problematic_patno=lexical_df.loc[lexical_df.isna().any(axis=1),'PATNO'].to_list()
cols_con_na = lexical_df.columns[lexical_df.isna().any()].to_list()


for patno in problematic_patno:
    mask = lexical_df['PATNO'] == patno
    for col in cols_con_na:
        valores_originales = lexical_df.loc[mask, col].tolist()
        valores_rellenos = fill_missing_values([float(v) if pd.notna(v) else np.nan for v in valores_originales])
        lexical_df.loc[mask, col] = valores_rellenos

lexical_df.isna().sum()

PATNO                           0
Visit ID                        0
Lexical Fluency Scaled Score    0
dtype: int64

# ANALYSIS OF PROGRESSION

In [48]:
progression_csv_upgrade(moca_df,prefix='MOCA')
progression_multi_csv_upgrade(moca_df,prefix='MOCA')
visit_csv_upgrade(moca_df,prefix='MOCA')
moca_df.to_csv('/Users/fserracrespi/Desktop/PD_Project_UofL/PD_DATA/PD_CSV_CLEAN/FINAL DATA TEST CLEAN/FINAL_MoCA_12SEP2025.csv', index=False)
cols_asignacion2(moca_df,df_secundario='MOCA',df_main='NON_MOTOR')


progression_csv_upgrade(scopa_data,prefix='SCOPAAUT')
progression_multi_csv_upgrade(scopa_data,prefix='SCOPAAUT')
visit_csv_upgrade(scopa_data,prefix='SCOPAAUT')
scopa_data.to_csv('/Users/fserracrespi/Desktop/PD_Project_UofL/PD_DATA/PD_CSV_CLEAN/FINAL DATA TEST CLEAN/FINAL_SCOPA_AUT_12SEP2025.csv', index=False)
cols_asignacion2(scopa_data,df_secundario='SCOPAAUT',df_main='NON_MOTOR')

progression_csv_upgrade(ep_df,prefix='EPWORTH')
progression_multi_csv_upgrade(ep_df,prefix='EPWORTH')
visit_csv_upgrade(ep_df,prefix='EPWORTH')
ep_df.to_csv('/Users/fserracrespi/Desktop/PD_Project_UofL/PD_DATA/PD_CSV_CLEAN/FINAL DATA TEST CLEAN/FINAL_EPWORTH_12SEP2025.csv', index=False)
cols_asignacion2(ep_df,df_secundario='EPWORTH',df_main='NON_MOTOR')

progression_csv_upgrade(rem_df,prefix='REMSLEEP')
progression_multi_csv_upgrade(rem_df,prefix='REMSLEEP')
visit_csv_upgrade(rem_df,prefix='REMSLEEP')
rem_df.to_csv('/Users/fserracrespi/Desktop/PD_Project_UofL/PD_DATA/PD_CSV_CLEAN/FINAL DATA TEST CLEAN/FINAL_REMSLEEP_12SEP2025.csv', index=False)
cols_asignacion2(rem_df,df_secundario='REMSLEEP',df_main='NON_MOTOR')

progression_csv_upgrade(semantic_df,prefix='SEMANTIC')
progression_multi_csv_upgrade(semantic_df,prefix='SEMANTIC')
visit_csv_upgrade(semantic_df,prefix='SEMANTIC')
semantic_df.to_csv('/Users/fserracrespi/Desktop/PD_Project_UofL/PD_DATA/PD_CSV_CLEAN/FINAL DATA TEST CLEAN/FINAL_SEMANTIC_12SEP2025.csv', index=False)
cols_asignacion2(semantic_df,df_secundario='SEMANTIC',df_main='NON_MOTOR')

progression_csv_upgrade(letter_df,prefix='LETTER')
progression_multi_csv_upgrade(letter_df,prefix='LETTER')
visit_csv_upgrade(letter_df,prefix='LETTER')
letter_df.to_csv('/Users/fserracrespi/Desktop/PD_Project_UofL/PD_DATA/PD_CSV_CLEAN/FINAL DATA TEST CLEAN/FINAL_LETTER_12SEP2025.csv', index=False)
cols_asignacion2(letter_df,df_secundario='LETTER',df_main='NON_MOTOR')

progression_csv_upgrade(benton_df,prefix='BENTONJLO')
progression_multi_csv_upgrade(benton_df,prefix='BENTONJLO')
visit_csv_upgrade(benton_df,prefix='BENTONJLO')
benton_df.to_csv('/Users/fserracrespi/Desktop/PD_Project_UofL/PD_DATA/PD_CSV_CLEAN/FINAL DATA TEST CLEAN/FINAL_BENTONJLO_12SEP2025.csv', index=False)
cols_asignacion2(benton_df,df_secundario='BENTONJLO',df_main='NON_MOTOR')


progression_csv_upgrade(symbol_df,prefix='SYMBOLDIGIT')
progression_multi_csv_upgrade(symbol_df,prefix='SYMBOLDIGIT')
visit_csv_upgrade(symbol_df,prefix='SYMBOLDIGIT')
symbol_df.to_csv('/Users/fserracrespi/Desktop/PD_Project_UofL/PD_DATA/PD_CSV_CLEAN/FINAL DATA TEST CLEAN/FINAL_SYMBOLDIGIT_12SEP2025.csv', index=False)
cols_asignacion2(symbol_df,df_secundario='SYMBOLDIGIT',df_main='NON_MOTOR')

progression_csv_upgrade(hopkins_df,prefix='HOPKINS')
progression_multi_csv_upgrade(hopkins_df,prefix='HOPKINS')
visit_csv_upgrade(hopkins_df,prefix='HOPKINS')
hopkins_df.to_csv('/Users/fserracrespi/Desktop/PD_Project_UofL/PD_DATA/PD_CSV_CLEAN/FINAL DATA TEST CLEAN/FINAL_HOPKINS_12SEP2025.csv', index=False)
cols_asignacion2(hopkins_df,df_secundario='HOPKINS',df_main='NON_MOTOR')

progression_csv_upgrade(anxiety_df,prefix='ANXIETY')
progression_multi_csv_upgrade(anxiety_df,prefix='ANXIETY')
visit_csv_upgrade(anxiety_df,prefix='ANXIETY')
anxiety_df.to_csv('/Users/fserracrespi/Desktop/PD_Project_UofL/PD_DATA/PD_CSV_CLEAN/FINAL DATA TEST CLEAN/FINAL_ANXIETY_12SEP2025.csv', index=False)
cols_asignacion2(anxiety_df,df_secundario='ANXIETY',df_main='NON_MOTOR')

progression_csv_upgrade(cog_df,prefix='COGNITIVECAT')
progression_multi_csv_upgrade(cog_df,prefix='COGNITIVECAT')
visit_csv_upgrade(cog_df,prefix='COGNITIVECAT')
cog_df.to_csv('/Users/fserracrespi/Desktop/PD_Project_UofL/PD_DATA/PD_CSV_CLEAN/FINAL DATA TEST CLEAN/FINAL_COGNITIVECAT_12SEP2025.csv', index=False)
cols_asignacion2(cog_df,df_secundario='COGNITIVECAT',df_main='NON_MOTOR')

progression_csv_upgrade(geriatic_df,prefix='GERIATRICDEPRESSION')
progression_multi_csv_upgrade(geriatic_df,prefix='GERIATRICDEPRESSION')
visit_csv_upgrade(geriatic_df,prefix='GERIATRICDEPRESSION')
geriatic_df.to_csv('/Users/fserracrespi/Desktop/PD_Project_UofL/PD_DATA/PD_CSV_CLEAN/FINAL DATA TEST CLEAN/FINAL_GERIATRICDEPRESSION_12SEP2025.csv', index=False)
cols_asignacion2(geriatic_df,df_secundario='GERIATRICDEPRESSION',df_main='NON_MOTOR')

progression_csv_upgrade(lexical_df,prefix='LEXICALFLUENCY')
progression_multi_csv_upgrade(lexical_df,prefix='LEXICALFLUENCY')
visit_csv_upgrade(lexical_df,prefix='LEXICALFLUENCY')
lexical_df.to_csv('/Users/fserracrespi/Desktop/PD_Project_UofL/PD_DATA/PD_CSV_CLEAN/FINAL DATA TEST CLEAN/FINAL_LEXICALFLUENCY_12SEP2025.csv', index=False)
cols_asignacion2(lexical_df,df_secundario='LEXICALFLUENCY',df_main='NON_MOTOR')



/Users/fserracrespi/Desktop/PD_PROJECT_UOFL/utils/Prog_df.py:59: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  p1.fillna(False, inplace=True)
/Users/fserracrespi/Desktop/PD_PROJECT_UOFL/utils/Prog_df.py:65: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  p2.fillna(False, inplace=True)
/Users/fserracrespi/Desktop/PD_PROJECT_UOFL/utils/Prog_df.py:71: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `

Progression CSV files updated successfully.
MULTI Progression CSV files updated successfully.
Visits CSV files updated successfully.
Progression CSV files updated successfully.


/Users/fserracrespi/Desktop/PD_PROJECT_UOFL/utils/Prog_df.py:193: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  p1.fillna(False, inplace=True)
/Users/fserracrespi/Desktop/PD_PROJECT_UOFL/utils/Prog_df.py:199: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  p2.fillna(False, inplace=True)
/Users/fserracrespi/Desktop/PD_PROJECT_UOFL/utils/Prog_df.py:205: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, se

MULTI Progression CSV files updated successfully.
Visits CSV files updated successfully.
Progression CSV files updated successfully.
MULTI Progression CSV files updated successfully.
Visits CSV files updated successfully.


/Users/fserracrespi/Desktop/PD_PROJECT_UOFL/utils/Prog_df.py:265: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  pV06.fillna(False, inplace=True)
/Users/fserracrespi/Desktop/PD_PROJECT_UOFL/utils/Prog_df.py:271: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  pV08.fillna(False, inplace=True)
/Users/fserracrespi/Desktop/PD_PROJECT_UOFL/utils/Prog_df.py:277: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior

...

MULTI Progression CSV files updated successfully.
Visits CSV files updated successfully.
Progression CSV files updated successfully.


/Users/fserracrespi/Desktop/PD_PROJECT_UOFL/utils/Prog_df.py:59: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  p1.fillna(False, inplace=True)
/Users/fserracrespi/Desktop/PD_PROJECT_UOFL/utils/Prog_df.py:65: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  p2.fillna(False, inplace=True)
/Users/fserracrespi/Desktop/PD_PROJECT_UOFL/utils/Prog_df.py:71: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `

MULTI Progression CSV files updated successfully.
Visits CSV files updated successfully.
Progression CSV files updated successfully.
MULTI Progression CSV files updated successfully.
Visits CSV files updated successfully.


/Users/fserracrespi/Desktop/PD_PROJECT_UOFL/utils/Prog_df.py:59: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  p1.fillna(False, inplace=True)
/Users/fserracrespi/Desktop/PD_PROJECT_UOFL/utils/Prog_df.py:65: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  p2.fillna(False, inplace=True)
/Users/fserracrespi/Desktop/PD_PROJECT_UOFL/utils/Prog_df.py:71: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `

,df_main,df_secundario,df_secundario_col
0,SUBJECT,Subject_Demographics_SocioEconomics_FHistory_A...,PATNO
1,SUBJECT,Subject_Demographics_SocioEconomics_FHistory_A...,Sporadic PD at Enrollment
2,SUBJECT,Subject_Demographics_SocioEconomics_FHistory_A...,RBD at Enrollment
3,SUBJECT,Subject_Demographics_SocioEconomics_FHistory_A...,Pink1 Mutation at Enrollment
4,SUBJECT,Subject_Demographics_SocioEconomics_FHistory_A...,Parkin Mutation at Enrollment
...,...,...,...
975,NON_MOTOR,GERIATRICDEPRESSION,GDS Short Score
976,NON_MOTOR,GERIATRICDEPRESSION,Has Depression
977,NON_MOTOR,LEXICALFLUENCY,PATNO
978,NON_MOTOR,LEXICALFLUENCY,Visit ID
